In [ ]:
import h5py
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import os
import gc 


BASE_PATH = '/kaggle/input/stanford-earthquake-dataset-stead/'
csv_path = os.path.join(BASE_PATH, 'merge.csv')
hdf5_path = os.path.join(BASE_PATH, 'merge.hdf5')

print("Loading Metadata...")
df = pd.read_csv(csv_path)


eq_samples = df[df['trace_category'] == 'earthquake_local'].sample(5000)
noise_samples = df[df['trace_category'] == 'noise'].sample(5000)
combined = pd.concat([eq_samples, noise_samples])


del df, eq_samples, noise_samples
gc.collect()

X = []
Y = []

print("Extracting 10,000 Seismograms from HDF5...")
with h5py.File(hdf5_path, 'r') as f:
    for i, (index, row) in enumerate(combined.iterrows()):
        trace_name = row['trace_name']
        
        data = f['data'][trace_name][:1000, :] 
        
        # Z-score Normalization
        std_val = np.std(data)
        if std_val == 0: std_val = 1e-6 
        data = (data - np.mean(data)) / std_val
        
        X.append(data.astype(np.float32))
        Y.append(1 if row['trace_category'] == 'earthquake_local' else 0)
        
        if (i+1) % 500 == 0:
            print(f"Processed {i+1}/10000 samples...")

X = np.array(X)
Y = np.array(Y)


X.tofile('samples.bin')
print("samples.bin created successfully!")


print("Starting Model Training...")
model = tf.keras.Sequential([
    layers.Input(shape=(1000, 3)),
    layers.Conv1D(16, 3, activation='relu'), 
    layers.MaxPooling1D(2),
    layers.Conv1D(8, 3, activation='relu'), 
    layers.Flatten(),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X, Y, epochs=20, batch_size=64, validation_split=0.2)


print("Starting INT8 Quantization...")


def representative_dataset():
    
    for i in range(200):
        sample = X[i:i+1].astype(np.float32)
        yield [sample]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]


quantized_tflite_model = converter.convert()


print("Generating C++ Header file...")
with open("model_data.h", "w") as f:
    f.write("#ifndef MODEL_DATA_H\n")
    f.write("#define MODEL_DATA_H\n\n")
    f.write("const unsigned char model_tflite[] = {\n")
    
    hex_array = [f"0x{b:02x}" for b in quantized_tflite_model]
    f.write(", ".join(hex_array))
    
    f.write("\n};\n\n")
    f.write(f"const int model_tflite_len = {len(quantized_tflite_model)};\n")
    f.write("#endif\n")

print("All Done! Please download 'samples.bin' and 'model_data.h' from the Output section.")